# Loan Approval Prediction Using Machine Learning

Compare K-nearest neighbors, logistic regression and random forest classifiers on loan approval data.

**Tools:** Python, pandas, scikit-learn

## 1. Import the Dataset

In [1]:

# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score


In [2]:

# Load the dataset
df = pd.read_excel('data/Loan_Approval_Dataset.xlsx')

# Display the first few rows
print(df.head())

# Display dataset shape
print("\nDataset Shape:", df.shape)


  Loan_Train Unnamed: 1 Unnamed: 2  Unnamed: 3    Unnamed: 4     Unnamed: 5  \
0    Loan_ID     Gender    Married  Dependents     Education  Self_Employed   
1   LP001002       Male         No           0      Graduate             No   
2   LP001003       Male        Yes           1      Graduate             No   
3   LP001005       Male        Yes           0      Graduate            Yes   
4   LP001006       Male        Yes           0  Not Graduate             No   

        Unnamed: 6         Unnamed: 7  Unnamed: 8        Unnamed: 9  \
0  ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term   
1             5849                  0         NaN               360   
2             4583               1508         128               360   
3             3000                  0          66               360   
4             2583               2358         120               360   

      Unnamed: 10    Unnamed: 11  Unnamed: 12  
0  Credit_History  Property_Area  Loan_Status  
1 

### Fixing the Column Names

In [3]:

# The first row contains the true column names.
# Reset the column names using the first row.

df.columns = df.iloc[0]
df = df[1:]

# Reset index
df.reset_index(drop=True, inplace=True)

# Display updated columns
print(df.columns)

# Display first rows
df.head()


Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object', name=0)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0,NaN,360,1,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508,128,360,1,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0,66,360,1,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358,120,360,1,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0,141,360,1,Urban,Y



## 2. Data Preparation

In [4]:

# Drop Loan_ID column
df = df.drop(columns=['Loan_ID'])

# Drop rows with missing values
df = df.dropna()

# Convert categorical variables to dummy variables
df = pd.get_dummies(df, drop_first=True)

# Display the processed dataset
print(df.head())

# Display shape after cleaning
print("\nCleaned Dataset Shape:", df.shape)


   Gender_Male  Married_Yes  Dependents_1  Dependents_2  Dependents_3  \
1         True         True          True         False         False   
2         True         True         False         False         False   
3         True         True         False         False         False   
4         True        False         False         False         False   
5         True         True         False          True         False   

   Education_Not Graduate  Self_Employed_Yes  ApplicantIncome_645  \
1                   False              False                False   
2                   False               True                False   
3                    True              False                False   
4                   False              False                False   
5                   False               True                False   

   ApplicantIncome_1000  ApplicantIncome_1025  ...  Loan_Amount_Term_120  \
1                 False                 False  ...                 Fal


## 3. Split the Data

In [5]:

# Define features and target
X = df.drop('Loan_Status_Y', axis=1)
y = df['Loan_Status_Y']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Display shapes
print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)


Training Features Shape: (384, 838)
Testing Features Shape: (96, 838)



## 4. Create a Pipeline with Min-Max Scaling and KNN

In [6]:

# Create pipeline
knn_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('knn', KNeighborsClassifier())
])

# Display pipeline
print(knn_pipeline)


Pipeline(steps=[('scaler', MinMaxScaler()), ('knn', KNeighborsClassifier())])



## 5. Fit a Default KNN Classifier

In [7]:

# Fit default KNN model
knn_pipeline.fit(X_train, y_train)

# Make predictions
y_pred_knn = knn_pipeline.predict(X_test)

# Calculate accuracy
default_knn_accuracy = accuracy_score(y_test, y_pred_knn)

print("Default KNN Accuracy:", round(default_knn_accuracy, 4))


Default KNN Accuracy: 0.7708



## 6. Create a Search Space for KNN

In [8]:

# Create search space
knn_search_space = {
    'knn__n_neighbors': range(1, 11)
}

print(knn_search_space)


{'knn__n_neighbors': range(1, 11)}



## 7. Perform Grid Search with 5-Fold Cross-Validation

In [9]:

# Perform grid search
knn_grid = GridSearchCV(
    knn_pipeline,
    knn_search_space,
    cv=5,
    scoring='accuracy'
)

# Fit the grid search
knn_grid.fit(X_train, y_train)

# Display best parameters
print("Best KNN Parameters:", knn_grid.best_params_)

# Display best cross-validation score
print("Best Cross-Validation Accuracy:", round(knn_grid.best_score_, 4))


Best KNN Parameters: {'knn__n_neighbors': 9}
Best Cross-Validation Accuracy: 0.7292



## 8. Evaluate the Best KNN Model

In [10]:

# Best KNN model
best_knn_model = knn_grid.best_estimator_

# Predictions
best_knn_predictions = best_knn_model.predict(X_test)

# Accuracy
best_knn_accuracy = accuracy_score(y_test, best_knn_predictions)

print("Best KNN Test Accuracy:", round(best_knn_accuracy, 4))


Best KNN Test Accuracy: 0.75



## 9. Expand the Search Space to Include Logistic Regression and Random Forest

In [11]:

# Create a general pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', KNeighborsClassifier())
])

# Create search space for multiple models
search_space = [
    {
        'classifier': [KNeighborsClassifier()],
        'classifier__n_neighbors': range(1, 11)
    },
    {
        'classifier': [LogisticRegression(max_iter=1000)],
        'classifier__C': [0.01, 0.1, 1, 10]
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 5, 10]
    }
]

# Perform grid search
grid_search = GridSearchCV(
    pipeline,
    search_space,
    cv=5,
    scoring='accuracy'
)

# Fit grid search
grid_search.fit(X_train, y_train)

# Display best model and parameters
print("Best Model:", grid_search.best_estimator_)
print("\nBest Parameters:", grid_search.best_params_)
print("\nBest Cross-Validation Accuracy:", round(grid_search.best_score_, 4))


Best Model: Pipeline(steps=[('scaler', MinMaxScaler()),
                ('classifier', LogisticRegression(C=1, max_iter=1000))])

Best Parameters: {'classifier': LogisticRegression(max_iter=1000), 'classifier__C': 1}

Best Cross-Validation Accuracy: 0.8075



## 10. Evaluate the Best Overall Model


In [12]:

# Best overall model
best_model = grid_search.best_estimator_

# Predictions
final_predictions = best_model.predict(X_test)

# Final accuracy
final_accuracy = accuracy_score(y_test, final_predictions)

print("Final Test Accuracy:", round(final_accuracy, 4))


Final Test Accuracy: 0.8125



## 11. Summary

This project used machine learning pipelines and grid search to predict loan approval outcomes.
